In [ ]:
#| default_exp testing

# testing

> Doubles, so code that uses anya can be tested with no runtime and no weights.

`FakeModel` answers from a script rather than a graph. `fake_segment` builds a segmenter whose label
map is the rectangles you name, which is what an editing pipeline downstream needs to be tested
against: a label map, class shares and boxes, with no wheel to install and nothing to download.

In [ ]:
#| export
from __future__ import annotations

import numpy as np
from fastcore.all import L

from anya.core import Model, Pred, Preds, item_src, items
from anya.vision import Prep

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp

In [ ]:
#| export
class FakeModel(Model):
    'A model that answers from a script, for testing a pipeline with no runtime installed.'
    _runtime = 'fake'

    def __init__(self,
                 answer=None,          # a dict of payload keys, or `f(item) -> dict`
                 task:str='classify',
                 labels=None,
                 name:str='fake',
                 **kw):
        self._setup(name, task=task, labels=labels, **kw)
        self._prep = Prep(size=None)
        self.answer = answer or {}

    def _read_spec(self): pass
    def _infer(self, x): raise NotImplementedError('a fake model has no graph to run')

    def predict(self, o, **kw) -> Pred:
        'The scripted answer, with this item named as its source.'
        a = self.answer(o) if callable(self.answer) else dict(self.answer)
        return Pred(dict(src=item_src(o), task=self.task, model=self.name), **a)

    def predict_all(self, o, bs:int=None, on_error:str='skip', types:str=None, exclude=None, **kw) -> Preds:
        'The same answer for every item the folder holds.'
        return Preds([self.predict(x, **kw) for x in items(o, types=types or self.modality, exclude=exclude)])

def fake_segment(blocks:dict,           # {'label': (x1, y1, x2, y2)}, painted in the order given
                 size:tuple=(64, 64),   # (width, height) of the label map
                 bg:str='background'    # the label for class 0
                ) -> FakeModel:
    'A segmenter whose label map is the rectangles you name. Class 0 is whatever is left.'
    labels = [bg] + list(blocks)
    m = np.zeros((size[1], size[0]), np.int32)
    for i, (k, b) in enumerate(blocks.items(), 1): m[b[1]:b[3], b[0]:b[2]] = i
    ids, cnt = np.unique(m, return_counts=True)
    cls = [dict(label=labels[int(i)], index=int(i), frac=round(float(c/m.size), 5))
           for i, c in sorted(zip(ids, cnt), key=lambda t: -t[1])]
    return FakeModel(dict(mask=m, shape=list(m.shape), classes=cls), task='segment', labels=labels)

def fake_detect(objects:list,           # [{'label', 'score', 'box'}], the boxes a detector would return
                labels=None) -> FakeModel:
    'A detector that returns the boxes you give it.'
    return FakeModel(dict(objects=list(objects)), task='detect',
                     labels=labels or sorted({o['label'] for o in objects}))

In [ ]:
#| hide
from anya.tasks import ask, class_boxes, detect, save_labelmap, save_masks, segment

_x = np.zeros((64, 64, 3), np.uint8)
m = fake_segment(dict(fence=(10, 40, 54, 50), car=(20, 20, 50, 45)))
test_eq((m.task, m.runtime, m.max_bs), ('segment', 'fake', 1))
p = segment(_x, m)
test_eq(p['shape'], [64, 64])
test_eq(p.label, 'background')                                  # the biggest class, as ever
test_eq({c['label']: c['index'] for c in p['classes']}['car'], 2)
test_eq(class_boxes(p, want='car')[0]['bbox'], [20, 20, 50, 45])
test_eq(class_boxes(p, want='fence')[0]['bbox'], [10, 40, 54, 50])
_d = mkdtemp()
test_eq(len(save_masks(p, _d, stem='x')), 3)
test_eq(save_labelmap(p, _d, stem='x').endswith('x.labels.png'), True)
test_eq(bool(ask(_x, dict(find='fence', task='segment', model=m))['hit'].any()), True)

_c = FakeModel(dict(preds=[dict(label='green', score=0.9)]))
test_eq(_c(_x).label, 'green')
test_eq(_c(_x).score, 0.9)
_det = fake_detect([dict(label='car', score=0.8, box=[1, 2, 3, 4])])
test_eq(detect(_x, _det).objects[0]['box'], [1, 2, 3, 4])
test_fail(lambda: _det._infer(None), contains='no graph')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()